# ARO 3D Studio — real TripoSR inference through the Studio (Colab)

**What this notebook proves when it succeeds:** real user images → the Studio's own upload/validation → persistent batch generation jobs → the **actual TripoSR model** (pinned revision, SHA-256-verified weights, safe loading) → raw assets → viewer preview → automatic preparation → validation → verified GLB export, for **three different object categories**, plus an invalid-file probe proving that one bad image does not stop its valid siblings.

**What it does not prove:** that outputs look good (a person must look and fill in the review), game-readiness, or anything about Unity 6. No mock, test double or procedural TITAN asset is used: the smoke script runs in `real_engine` mode and refuses to continue if the real engine is not runnable.

**Before you start:** Runtime ▸ Change runtime type ▸ **T4 GPU**. Run the cells **in order**, one at a time. Nothing here needs a secret or token (the model and the example images are public). If a cell fails, run the last cell ("Failure bundle") and send me the downloaded `aro-logs.zip` plus a screenshot of the failing cell.

## 1 · GPU and environment check
**Expected:** a table with `Tesla T4` (≈ 15 GB), `torch … cuda True`, free disk ≥ 20 GB. **If `cuda False`:** stop — the runtime is not a GPU runtime.

In [ ]:
import os, sys, platform, shutil
os.makedirs('/content/logs', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| cuda runtime', torch.version.cuda)
print('free disk GB:', round(shutil.disk_usage('/content').free/1e9, 1))
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then restart and run again'

## 2 · Get the Studio at a known revision
**Expected:** the last line prints the commit hash you are testing (`STUDIO_REF` below; `main` branch name or a commit). Keep that hash: it goes into the evidence. `pip` output is saved to `/content/logs/pip-studio.log`; on failure its tail is shown.

In [ ]:
STUDIO_REF = 'claude/zen-johnson-je3dzk'   # branch or full commit hash to verify
if not os.path.isdir('/content/aro'):
    !git clone -q https://github.com/AffanNamu/ARO-3D-STUDIO /content/aro
%cd /content/aro
changed = !git status --porcelain --untracked-files=no
assert not changed, f'/content/aro has local changes that a checkout could overwrite: {changed} - save or remove them first'
!git fetch -q origin
is_branch = os.system(f'git show-ref --verify --quiet refs/remotes/origin/{STUDIO_REF}') == 0
if is_branch:
    !git checkout -q -B $STUDIO_REF origin/$STUDIO_REF     # an EXISTING clone is moved to the current branch tip (never left on a stale commit)
else:
    !git checkout -q $STUDIO_REF
!git log -1 --format='STUDIO COMMIT %H  %s'
!pip install -q -r requirements.txt > /content/logs/pip-studio.log 2>&1 || (tail -40 /content/logs/pip-studio.log; false)
print('studio requirements installed (already-satisfied packages are left alone)')

## 3 · TripoSR source at the commit pinned in the lock, and its dependencies
**Expected:** `TSR COMMIT 107cefdc…` and `byte-identical to the pinned commit`; a list of which packages are already importable; only the missing ones are installed (log: `/content/logs/pip-triposr.log`); then the **`torchmcubes`** build (3–10 min, log: `/content/logs/pip-torchmcubes.log`) ending with `torchmcubes import OK` and whether it has CUDA kernels.

**Why `torchmcubes` needs special handling:** its current `main` branch refuses to build unless PyTorch is already installed (pip's default isolated build fails at *"Preparing metadata"*), so the documented procedure is used: install the build tools, then `pip install --no-build-isolation`. Because this runtime's torch is a **CUDA build**, torch's own CMake config requires a CUDA compiler, so `torchmcubes` is built **with CUDA kernels** (`nvcc` 13.0 must match torch's CUDA 13), for your GPU's compute capability only. It takes **about 3–8 minutes and the output is shown live — please do not interrupt it.** It is built from the repository's unpinned `main`; the exact commit built is recorded in the evidence.
**Import checks run in a fresh Python process**, so a package that pip upgraded (numpy/numba) cannot give a false "import failed" from this kernel's stale memory. If a later cell still reports a numpy error, use Runtime ▸ Restart session and re-run from cell 2.

In [ ]:
import json, re, hashlib, subprocess, shutil
lock = json.load(open('aro_studio/engines/locks/triposr.json'))
TSR_COMMIT = lock['code']['commit']
if not os.path.isdir('/content/TripoSR'):
    !git clone -q https://github.com/VAST-AI-Research/TripoSR /content/TripoSR
cur = !git -C /content/TripoSR rev-parse HEAD
if cur[0] != TSR_COMMIT:
    !git -C /content/TripoSR checkout -q $TSR_COMMIT
head = !git -C /content/TripoSR rev-parse HEAD
print('TSR COMMIT', head[0]); assert head[0] == TSR_COMMIT, 'TripoSR checkout does not match the pinned commit'
mods = !git -C /content/TripoSR status --porcelain --untracked-files=no
if mods:   # e.g. a local compatibility patch: NOT discarded, but recorded in the evidence (file list + diff hash)
    diff_lines = !git -C /content/TripoSR diff
    diff = '\n'.join(diff_lines)
    print('NOTE: /content/TripoSR differs from the pinned commit (local modifications kept):', mods)
    print('      diff sha256:', hashlib.sha256(diff.encode()).hexdigest())
else:
    print('TripoSR source is byte-identical to the pinned commit')
os.environ['PYTHONPATH'] = '/content/TripoSR'
sys.path.insert(0, '/content/TripoSR')

ENV = {**os.environ, 'PYTHONPATH': '/content/TripoSR:/content/aro'}
def importable(mod):   # fresh process: reflects what the Studio will see, not this kernel's stale imports
    return subprocess.run([sys.executable, '-c', f'import {mod}'], capture_output=True, env=ENV).returncode == 0

IMPORT_NAME = {'omegaconf': 'omegaconf', 'einops': 'einops', 'transformers': 'transformers', 'huggingface-hub': 'huggingface_hub', 'rembg': 'rembg', 'onnxruntime': 'onnxruntime'}
pins = {re.split(r'[=<>~! ]', l.split('#')[0].strip())[0]: l.split('#')[0].strip() for l in open('requirements-triposr.txt') if l.strip() and not l.startswith(('#', 'git+', 'torch'))}
ok_now = {n: importable(m) for n, m in IMPORT_NAME.items()}
missing = [pins[n] for n, ok in ok_now.items() if not ok and n in pins]
print('already importable:', [n for n, ok in ok_now.items() if ok]); print('will install      :', missing or 'nothing')
if missing:
    !pip install -q {' '.join(missing)} > /content/logs/pip-triposr.log 2>&1 || (tail -60 /content/logs/pip-triposr.log; false)

LOG = '/content/logs/pip-torchmcubes.log'
import threading, queue, time

def stream(cmd, env=None, limit_s=1800, hide=('Requirement already satisfied',)):
    """Run a command, print its output LIVE (and keep it in the log), with a heartbeat when it is quiet. Never hides progress."""
    q = queue.Queue(); log = open(LOG, 'a'); log.write(f"\n$ {' '.join(cmd)}\n")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env or os.environ, bufsize=1)
    def pump():
        for l in p.stdout: q.put(l)
        q.put(None)                      # end-of-output marker
    threading.Thread(target=pump, daemon=True).start()
    t0 = time.time()
    while True:
        try:
            line = q.get(timeout=20)
        except queue.Empty:
            print(f'  ... still working ({int(time.time() - t0)} s) - do NOT interrupt'); line = ''
        if line is None: break
        if line:
            log.write(line)
            if not line.lstrip().startswith(hide): print(line.rstrip()[:220])
        if time.time() - t0 > limit_s: p.kill(); print('TIMEOUT after', limit_s, 's'); log.write('TIMEOUT\n'); return False
    rc = p.wait(); log.write(f'-> exit {rc}\n'); log.close(); return rc == 0

BUILD = [sys.executable, '-m', 'pip', 'install', '-v', '--no-build-isolation', 'git+https://github.com/tatsy/torchmcubes.git']

if importable('torchmcubes'):
    print('torchmcubes is already importable -> REUSED, not rebuilt')
else:
    # torch here is a CUDA build, so torch's own CMake config REQUIRES a CUDA compiler: a CPU-only build is not possible. The Colab
    # toolkit (nvcc) must match torch's CUDA major version; the arch is taken from the GPU so nothing is compiled for other cards.
    nvcc = shutil.which('nvcc'); print('nvcc:', nvcc or 'NOT FOUND'); assert nvcc, 'no CUDA compiler (nvcc) in this runtime: send me this message'
    import torch
    print(subprocess.run([nvcc, '--version'], capture_output=True, text=True).stdout.strip().splitlines()[-1], '| torch built for CUDA', torch.version.cuda)
    cap = subprocess.run([sys.executable, '-c', 'import torch; print("%d%d" % torch.cuda.get_device_capability())'], capture_output=True, text=True).stdout.strip()
    assert cap.isdigit(), 'could not read the GPU compute capability'
    print(f'Building torchmcubes WITH CUDA kernels for compute capability {cap} (about 3-8 minutes; output is live below; do not interrupt).')
    open(LOG, 'w').write('')
    assert stream([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-build-core>=1.0', 'pybind11>=2.10', 'ninja', 'cmake>=3.24', 'setuptools', 'wheel']), 'could not install the build tools (see ' + LOG + ')'
    if not stream(BUILD, {**os.environ, 'CMAKE_ARGS': f'-DCMAKE_CUDA_ARCHITECTURES={cap}'}):
        print(''.join(open(LOG).readlines()[-150:])); raise RuntimeError('torchmcubes could not be built: send /content/logs/pip-torchmcubes.log')
r = subprocess.run([sys.executable, '-c', 'import torchmcubes; print("torchmcubes import OK | CUDA kernels:", torchmcubes.HAS_CUDA)'], capture_output=True, text=True, env=ENV)
print(r.stdout.strip() or r.stderr[-1500:]); assert r.returncode == 0, 'torchmcubes still not importable'
import torch; assert tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) >= (2, 6), 'torch >= 2.6.0 is required (weights_only safe loading)'

## 4 · Stage 2 — model / weight / licence verification (before any inference)
**Expected, in order:** the package list (all importable, `torchmcubes` included); `VIT_COMPAT_OK` and `TSR_IMPORT_OK` (a quick check, with a tiny random model and **no download**, that the installed `transformers` works with TripoSR's DINO tokenizer — this runtime ships a much newer `transformers` than TripoSR was written for); `lock missing: []`, `licence status: verified`, `availability ok: True`; then `OK: all files match the lock` after the ≈ 1.7 GB download at the pinned revision (`5b521936…`). Every file's SHA-256 is compared with the lock **before** anything is deserialised; the loader then uses `torch.load(weights_only=True)`, a class allow-list and a pinned DINO config.
**If `VIT_COMPAT_OK` is missing:** the printed traceback is the blocker — run the optional **4b** cell below, or send me the traceback. **If `availability ok: False`:** the printed `reasons` are the exact blocker; do not edit the lock.
**Background removal:** photos with busy backgrounds use neural matting (`rembg`; it downloads a ≈ 176 MB U²-Net ONNX model on first use that is **not** hash-pinned — its SHA-256 is recorded in the evidence).

In [ ]:
import json, subprocess
def version_of(mod):
    r = subprocess.run([sys.executable, '-c', f'import {mod} as m; print(getattr(m, "__version__", "ok"))'], capture_output=True, text=True, env=ENV)
    return r.stdout.strip() if r.returncode == 0 else 'IMPORT FAILED: ' + (r.stderr.strip().splitlines() or ['?'])[-1][:160]
for m in ('torch', 'transformers', 'omegaconf', 'einops', 'huggingface_hub', 'torchmcubes', 'tsr', 'trimesh', 'rembg', 'onnxruntime', 'numpy', 'PIL'):
    print(f'{m:16s}', version_of(m))

SMOKE = '''
import torch
from transformers import ViTConfig
from transformers.models.vit.modeling_vit import ViTModel
m = ViTModel(ViTConfig(hidden_size=32, num_hidden_layers=1, num_attention_heads=2, intermediate_size=64, image_size=224, patch_size=16))
o = m(torch.zeros(1, 3, 224, 224), interpolate_pos_encoding=True)
print("VIT_COMPAT_OK", tuple(o.last_hidden_state.shape), o.pooler_output is not None)
import tsr.system, tsr.models.tokenizers.image
print("TSR_IMPORT_OK")
'''
r = subprocess.run([sys.executable, '-c', SMOKE], capture_output=True, text=True, env=ENV); print(r.stdout, r.stderr[-2500:])
assert 'TSR_IMPORT_OK' in r.stdout, 'transformers/TripoSR compatibility check failed: see the traceback above (optional cell 4b, or send it to me)'

code_ = '''
import json
from aro_studio.engines.triposr import TripoSREngine
from aro_studio.engines.weights import load_lock
e = TripoSREngine(); av = e.availability("cuda")
print("lock missing:", load_lock("triposr").missing())
print("licence status:", e.info.license.status, "| model version:", e.info.version)
print("availability ok:", av.ok); print(json.dumps(av.to_dict(), indent=1))
'''
r = subprocess.run([sys.executable, '-c', code_], capture_output=True, text=True, env=ENV); print(r.stdout, r.stderr[-2000:])
assert 'availability ok: True' in r.stdout and 'licence status: verified' in r.stdout, 'Engine not runnable: see the reasons above'
!python scripts/verify_triposr_weights.py 2>&1 | tail -12
print('rembg:', 'available - neural matting will cut out busy backgrounds' if importable('rembg') else 'NOT importable - a photo with a busy background would be sent to the model UNCUT; send me the log of cell 3')

## 4b · OPTIONAL — only if `VIT_COMPAT_OK` was missing
TripoSR's exact pin (`transformers==4.35.0`) cannot be installed on Python 3.13 without compiling Rust (its `tokenizers` has no 3.13 wheel). If the very new preinstalled `transformers` is incompatible with TripoSR's DINO tokenizer, this cell installs the nearest series that *does* have Python 3.13 wheels (`transformers==4.46.3`, `tokenizers==0.20.3`, `huggingface-hub<1`). It is a deviation from TripoSR's own pin and is recorded in the evidence (package versions). Then re-run cell 4.

In [ ]:
APPLY_TRANSFORMERS_FALLBACK = False     # set True only if cell 4 failed its compatibility check
if APPLY_TRANSFORMERS_FALLBACK:
    !pip install -q "transformers==4.46.3" "tokenizers==0.20.3" "huggingface-hub>=0.23.2,<1.0" > /content/logs/pip-transformers-fallback.log 2>&1 || (tail -40 /content/logs/pip-transformers-fallback.log; false)
    print('installed the transformers 4.46.3 series; now re-run cell 4')
else:
    print('skipped (only needed if cell 4 failed its compatibility check)')

## 5 · Khronos glTF validator (optional but recommended)
**Expected:** no output. Without it the export check still runs, but the Khronos result is recorded as "not run".

In [ ]:
!mkdir -p /content/gv && cd /content/gv && npm init -y >/dev/null 2>&1 && npm i gltf-validator >/dev/null 2>&1; ls /content/gv/node_modules | head -3
os.environ['ARO_GLTF_VALIDATOR_NODE_PATH'] = '/content/gv/node_modules'

## 6 · The input image(s)
### Stage 1 — image verification
**Expected:** the vehicle image is found at `/content/Luxury ARO 480 Heavy-Duty Truck.png` (or you are asked to upload one), it **opens**, and its size, mode and SHA-256 are printed. Nothing else is used for the truck run: no sample image, no procedural model, no earlier GLB.

Two modes (set `FOCUSED_TRUCK_ONLY` in the next cell):
* `True` — **focused run:** only your truck image goes through the real pipeline. The evidence will say **PARTIAL** because the acceptance rule needs ≥ 3 different images; the truck result itself is still fully recorded.
* `False` — **full run:** your truck + TripoSR's chair, teapot and robot examples (documented, SHA-256-pinned): the evidence can reach **PASS**.

In [ ]:
IMAGE_PATH = ''       # leave empty to auto-detect '/content/Luxury ARO 480 Heavy-Duty Truck.png', otherwise set the full path of YOUR image in /content
TRUCK = IMAGE_PATH or '/content/Luxury ARO 480 Heavy-Duty Truck.png'
if os.path.isfile(TRUCK):
    VEHICLE_FILE = TRUCK
    print('found the image in this runtime - NOT asking for an upload:', TRUCK)
else:
    from google.colab import files
    print('image not found in this runtime: upload it now (choose the truck image you want converted)')
    up = files.upload(); assert len(up) == 1, 'upload exactly one image'
    name = list(up)[0]
    VEHICLE_FILE = '/content/' + os.path.basename(name)
    with open(VEHICLE_FILE, 'wb') as fh: fh.write(up[name])     # files.upload() saves into the CURRENT directory (/content/aro), so write it to /content explicitly
    print('saved to', VEHICLE_FILE)
import hashlib
from PIL import Image
with Image.open(VEHICLE_FILE) as im:
    im.verify()                      # raises if the file is corrupt
with Image.open(VEHICLE_FILE) as im:
    im.load()
    print('STAGE 1 image verification: OPENED OK')
    print('  file  :', VEHICLE_FILE); print('  size  :', os.path.getsize(VEHICLE_FILE), 'bytes')
    print('  pixels:', im.size, '| mode', im.mode, '| format', im.format)
    print('  sha256:', hashlib.sha256(open(VEHICLE_FILE, 'rb').read()).hexdigest())
    VEHICLE_SIZE = im.size

In [ ]:
FOCUSED_TRUCK_ONLY = True            # True: only the truck (status will be PARTIAL); False: truck + 3 documented example objects
VEHICLE_CATEGORY = 'heavy_truck'
VEHICLE_NAME     = os.path.splitext(os.path.basename(VEHICLE_FILE))[0][:60]
VEHICLE_SOURCE   = (f"Project-owner-supplied image '{os.path.basename(VEHICLE_FILE)}' (provenance as stated by the project owner; "
                    "not a TripoSR sample, not the procedural TITAN 480 model)")
assert VEHICLE_SOURCE.strip(), 'State the owner/licence of the vehicle image in VEHICLE_SOURCE before continuing'

import json
EX = '/content/TripoSR/examples'
EXPECTED = {   # SHA-256 of the exact files inspected before this notebook was written
  'chair.png':  '2503c12a74419d91a4c6c9f1affc48fee6e2b8b9091956ca6211e91ada57b5bf',
  'teapot.png': '66b0b8f7dd759e97d20665999885e0548f96c95dadee6bf9762ccd877c596b8c',
  'robot.png':  '092ee61f1729ac9ed516c8d031d071f44dfa6e79f6ece04f32c988721b2b657a',
}
SRC = 'TripoSR repository examples/ at commit ' + TSR_COMMIT[:12] + ' (repo MIT-licensed; per-image authorship not stated upstream)'
items = [{'path': VEHICLE_FILE, 'category': VEHICLE_CATEGORY, 'name': VEHICLE_NAME, 'source': VEHICLE_SOURCE}]
if not FOCUSED_TRUCK_ONLY:
    for f, h in EXPECTED.items():
        got = hashlib.sha256(open(f'{EX}/{f}', 'rb').read()).hexdigest(); assert got == h, f'{f} differs from the inspected file ({got})'
    items += [
      {'path': f'{EX}/chair.png',  'category': 'furniture',        'name': 'Chair',     'source': SRC},
      {'path': f'{EX}/teapot.png', 'category': 'household_object', 'name': 'Teapot',    'source': SRC},
      {'path': f'{EX}/robot.png',  'category': 'equipment',        'name': 'Toy robot', 'source': SRC},
    ]
json.dump({'items': items}, open('/content/manifest.json', 'w'), indent=1)
from IPython.display import display
for it in items:
    im = Image.open(it['path']); t = im.copy(); t.thumbnail((260, 260))
    print(f"{it['name']:24s} {it['category']:17s} {im.size} {im.mode}  sha256 {hashlib.sha256(open(it['path'],'rb').read()).hexdigest()[:16]}…"); display(t)
print('mode:', 'FOCUSED (truck only, will be PARTIAL)' if FOCUSED_TRUCK_ONLY else 'FULL (4 images)')

## 7 · Stage 3 — run the real pipeline through the Studio
**Expected (≈ 10–25 minutes for four images, ≈ 5–10 for the truck alone):** the Studio starts, one batch request queues the image(s) plus a deliberately invalid text file (rejected while the valid image keeps going), the jobs run, and the JSON at the end shows `"mode": "real_engine"`. Full run: `"real_inference": "PASS"`. **Focused truck run: `"PARTIAL"` with a coverage note — that is the correct, honest status for one image.**
Per image the evidence records the model revision, device, input hash, job id, load/inference time, mesh statistics, output hash, preview, preparation steps, validation status and the re-opened GLB export.
**If it ends `BLOCKED`:** the engine was not runnable — the printed reasons are the blocker. **If `FAIL`:** it still wrote `smoke-evidence.json`; send it with the logs.

In [ ]:
OUT = '/content/aro/docs/acceptance/evidence/ai/smoke'
FLAG = '--allow-partial-coverage' if FOCUSED_TRUCK_ONLY else ''
!cd /content/aro && python scripts/real_inference_smoke.py --manifest /content/manifest.json --out $OUT --preset BALANCED $FLAG 2>&1 | tee /content/logs/smoke-run.log | tail -45

## 8 · Evidence check — and the five stages, each reported separately
**Expected:** the acceptance-report reader prints `PASS` (full run) or `PARTIAL` (focused run), then a stage table. A stage is shown as completed **only** when the recorded evidence proves it; otherwise it says `NOT COMPLETED` and why. The reader **rejects** test-double evidence, cache hits and missing hashes.

In [ ]:
import subprocess, textwrap
code_ = textwrap.dedent('''
    from pathlib import Path
    from aro_studio.verification.evidence import read_smoke_evidence
    e = read_smoke_evidence(Path("/content/aro/docs/acceptance/evidence/ai/smoke/smoke-evidence.json"))
    print(e.status, "-", e.detail)
    for i in e.facts.get("items", []): print("  ", i)
''')
print(subprocess.run([sys.executable, '-c', code_], capture_output=True, text=True, env={**os.environ, 'PYTHONPATH': '/content/aro'}).stdout)
ev = json.load(open(OUT + '/smoke-evidence.json'))
print('mode:', ev['mode'], '| studio commit:', ev['studio_commit'], '| GPU:', (ev['environment'].get('cuda') or {}).get('device'))
for i in ev['items']:
    ri = i.get('real_inference', {}); pr = i.get('preparation', {}); ex = i.get('export', {})
    print(f"{i['name']:14s} {i.get('result','?'):5s} tris={ri.get('triangles')} load={ri.get('model_load_time_s')}s infer={ri.get('inference_time_s')}s device={ri.get('device')} cache_hit={ri.get('cache_hit')} prep={((pr.get('validation') or {}).get('status'))} export={ex.get('status')}")
print('invalid-file probe:', ev.get('negative_probe', {}).get('result'))
print('\n================ STAGE REPORT (from the recorded evidence only) ================')
def stage(n, name, ok, detail): print(f"STAGE {n} {name:28s} {'COMPLETED' if ok else 'NOT COMPLETED'} - {detail}")
env = ev['environment']; eng = ev.get('engine', {})
REAL = ev['mode'] == 'real_engine' and eng.get('id') == 'triposr'
if not REAL: print('!!! NOT REAL INFERENCE: mode=%s engine=%s - stages 2, 3 and 4 cannot be COMPLETED from this evidence !!!' % (ev['mode'], eng.get('id')))
for i in ev['items']:
    ri, pr, ex = i.get('real_inference', {}), i.get('preparation', {}), i.get('export', {})
    img = i.get('input_image', {})
    print(f"\n--- {i['name']} ({i['category']}) ---")
    stage(1, 'image verification', bool(img) and ri.get('input_bytes_equal_stored_source') is True,
          f"{img.get('width')}x{img.get('height')} {img.get('mode')} {img.get('format')}, sha256 {i['input_sha256'][:16]}..., stored byte-identical to the source: {ri.get('input_bytes_equal_stored_source')}")
    stage(2, 'model/weights/licence', REAL and bool(ri.get('model_ran')) and eng.get('licence', {}).get('status') == 'verified' and not eng.get('lock_missing'),
          f"{eng.get('version')} | licence {eng.get('licence', {}).get('status')} | lock missing {eng.get('lock_missing')} | loader verified SHA-256 before deserialising (it refuses otherwise)")
    stage(3, 'real inference', REAL and bool(ri.get('model_ran')) and ri.get('cache_hit') is False,
          f"device={ri.get('device')} load={ri.get('model_load_time_s')}s infer={ri.get('inference_time_s')}s job={i.get('generation_job', {}).get('job_id')} state={i.get('generation_job', {}).get('state')} cache_hit={ri.get('cache_hit')} output_sha256={str(ri.get('output_sha256'))[:16]}...")
    rs = i.get('raw_sanity') or {}
    stage(4, 'mesh validation (raw)', REAL and bool(ri.get('finite')) and ri.get('triangles', 0) > 1000,
          f"{ri.get('vertices')} vertices, {ri.get('triangles')} triangles, finite={ri.get('finite')}, extents(native)={[round(x, 3) for x in ri.get('extents_native', [])]}, watertight={rs.get('watertight')}, components={rs.get('components')}, issues={[x['id'] for x in rs.get('issues', [])]}")
    v = pr.get('validation') or {}
    stage('4b', 'preparation + validation', pr.get('job_state') == 'COMPLETED' and pr.get('ok'),
          f"steps={[(s['step'].split(':')[0], s['status']) for s in pr.get('steps', [])]} | validation={v.get('status')} {v.get('overall')}% | raw immutable={i.get('raw_immutable')}")
    for n in pr.get('needs_human') or []: print('      needs a person:', n)
    stage(5, 'GLB export', ex.get('status') == 'COMPLETED' and ex.get('reopened_vertices_finite') is True and ex.get('sha256_matches_record') is True,
          f"{ex.get('size_bytes')} bytes, download hash matches record={ex.get('sha256_matches_record')}, re-opened: {ex.get('reopened_geometry_count')} geometries / {ex.get('reopened_triangles_all_lods')} triangles / finite={ex.get('reopened_vertices_finite')}, Khronos={ (ex.get('khronos') or {}).get('errors', 'not run') } errors")
print('\nrembg models used (SHA-256 recorded, not pinned):', env.get('rembg_models'))
print('overall status:', ev['status'])

## 9 · LOOK at the results (this is the quality check)
Each pair is the **raw TripoSR mesh** (left) and the **prepared mesh** (right), from three angles. Judge: is it recognisable? Are parts missing, fused, hollow or flat? Which way is up and which way is the front?

In [ ]:
import glob
from PIL import Image
from IPython.display import display
for n, i in enumerate(ev['items'], 1):
    if not i.get('raw_views'): print(n, i['name'], '- no output'); continue
    print(f"=== {n:02d} {i['name']} ({i['category']}) ===")
    row = []
    for stage in ('raw', 'prepared'):
        for view in ('front', 'left', '3q'):
            p = f"{OUT}/views/{n:02d}-{stage}-{view}.png"
            if os.path.exists(p): row.append(Image.open(p).resize((300, 300)))
    sheet = Image.new('RGB', (300 * 3, 300 * 2), (230, 230, 230))
    for k, im in enumerate(row): sheet.paste(im, ((k % 3) * 300, (k // 3) * 300))
    display(sheet)

## 10 · Record your review (only after looking at step 9)
Fill **every** field for every image that produced a mesh — the script **refuses** empty fields, unknown names, or evidence from a test double, and never writes text for you. `recognisable` must be `yes`, `partially` or `no`. Be specific and honest: "back of the chair is a flat sheet" is the kind of thing worth recording.

In [ ]:
REVIEWER = ''     # your name
reviews = {
  VEHICLE_NAME:     {'recognisable': '', 'visual_defects': '', 'category_limitations': '', 'orientation_up_front': ''},
  'Chair':         {'recognisable': '', 'visual_defects': '', 'category_limitations': '', 'orientation_up_front': ''},
  'Teapot':        {'recognisable': '', 'visual_defects': '', 'category_limitations': '', 'orientation_up_front': ''},
  'Toy robot':     {'recognisable': '', 'visual_defects': '', 'category_limitations': '', 'orientation_up_front': ''},
}
json.dump(reviews, open('/content/review.json', 'w'), indent=1)
!cd /content/aro && python scripts/fill_human_review.py --evidence $OUT/smoke-evidence.json --reviews /content/review.json --reviewer "$REVIEWER"

## 11 · Package the evidence
**Expected:** two zips are downloaded.
* `smoke-evidence-repo.zip` — JSON, report, original input images, renders, prepared GLBs: **safe to commit** under `docs/acceptance/evidence/ai/smoke/`.
* `smoke-raw-meshes.zip` — the untouched raw TripoSR meshes (large): **keep it (Drive), do not commit**; hashes are in the JSON.
Send me `smoke-evidence-repo.zip` (and `aro-logs.zip` if anything failed).

In [ ]:
!cd /content/aro && python scripts/package_smoke_evidence.py --dir $OUT --out /content/package
from google.colab import files
files.download('/content/package/smoke-evidence-repo.zip'); files.download('/content/package/smoke-raw-meshes.zip')

## 11b · OPTIONAL — send the evidence straight to the repository (a "connection" without zip files)
Colab cannot be remote-controlled by the assistant, but this cell lets the notebook push the **repo-safe** evidence to a **separate branch** (`evidence/smoke-<time>`), which the repository session then fetches. Skip it if you prefer to send the zip.

**One-time setup (the token never appears in the notebook or its output):**
1. GitHub ▸ Settings ▸ Developer settings ▸ *Fine-grained personal access tokens* ▸ Generate: **Only select repositories → `AffanNamu/ARO-3D-STUDIO`**, permission **Contents: Read and write**, expiry 7 days.
2. Colab ▸ left sidebar **🔑 Secrets** ▸ Add `GITHUB_TOKEN` = the token ▸ switch **Notebook access** on. Never paste it into a cell or chat.
3. Run the cell. It refuses to push to `main`, `claude/*` or the checked-out branch, refuses test-double evidence and files > 8 MB, stages only `docs/acceptance/evidence/ai/smoke/` (never `raw/`), and prints the branch name. Revoke the token afterwards.

In [ ]:
from google.colab import userdata
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')   # raises if the secret is missing or notebook access is off
!cd /content/aro && python scripts/push_smoke_evidence.py --repo /content/aro
del os.environ['GITHUB_TOKEN']

## 12 · Failure bundle (run this if ANY cell failed)
Bundles every log and the evidence JSON (no weights, no secrets) so I can diagnose without guessing.

In [ ]:
!cd /content && zip -qr aro-logs.zip logs aro/docs/acceptance/evidence/ai/smoke/smoke-evidence.json aro/docs/acceptance/evidence/ai/smoke/smoke-report.md 2>/dev/null; ls -l aro-logs.zip
from google.colab import files; files.download('/content/aro-logs.zip')